### Import libraries, read data and display head

In [16]:
import numpy as np
import pandas as pd

df = pd.read_csv("./DirtyData.csv")
df.head(20)

,first_name,last_name,email,gender,income,tax_15
0,Alverta,Colkett,acolkett0@cocolog-nifty.com,Female,123072.34,NaN
1,Nichole,Brandassi,nbrandassi1@mail.ru,Bigender,NaN,24115.00
2,Ruperto,Chaddock,rchaddock2@mail.ru,Male,62524.77,9378.72
3,Lula,Sorrill,lsorrill3@hatena.ne.jp,Female,48000.77,7200.12
4,Blondell,Benard,bbenard4@admin.ch,Female,88638.49,13295.77
5,Jacquelyn,Fawdry,jfawdry5@addthis.com,Female,NaN,14493.00
6,Kurt,Dugue,kdugue6@istockphoto.com,Male,105041.02,15756.15
7,Cordelie,Bloxsom,cbloxsom7@state.gov,Woman,236589.68,35488.45
8,Katinka,Renzo,krenzo8@thetimes.co.uk,Woman,194301.63,29145.24
9,Ketty,Pakeman,kpakeman9@chicagotribune.com,Female,239219.05,NaN


### Step 1: Inspect before touching anything

In [8]:
print("Shape:", df.shape)
print("Missing values per columns:")
print(df.isna().sum())

print(f"Negative incomes: {(df['income']<0).sum()}\n")
print(f"Gender categories and their counts: {df["gender"].value_counts()}")

Shape: (1000, 6)
Missing values per columns:
first_name      0
last_name       0
email           0
gender          0
income        115
tax_15         28
dtype: int64
Negative incomes: 86

Gender categories and their counts: gender
Female         469
Male           409
Non-binary      21
Genderfluid     19
Polygender      18
Genderqueer     17
Bigender        16
Agender         13
Man             10
Woman            3
Men              3
Women            2
Name: count, dtype: int64


### Step 2: Handle missing value by Reconstruction (not guessing)
Mean imputation would fill a missing income with the average income. But we can do better here, because tax_15 is always exactly 15% of income. So if income is missing but tax is present, we recover the exact income: income = tax/0.15. The reverse holds too: a missing tax is just income * 0.15.

This beats the mean: we use a known relationship between columns to recover the true value instead of guessing.

In [17]:
df["income"] = np.where(df["income"].isna() & df["tax_15"].notna(), df["tax_15"]/0.15, df["income"])

df["tax_15"] = np.where(df["tax_15"].isna() & df["income"].notna(), df["income"]*0.15, df["tax_15"])

print("Missing value after reconstruction:", df.isna().sum())
df.head(20)

Missing value after reconstruction: first_name    0
last_name     0
email         0
gender        0
income        0
tax_15        0
dtype: int64


,first_name,last_name,email,gender,income,tax_15
0,Alverta,Colkett,acolkett0@cocolog-nifty.com,Female,123072.340000,18460.8510
1,Nichole,Brandassi,nbrandassi1@mail.ru,Bigender,160766.666667,24115.0000
2,Ruperto,Chaddock,rchaddock2@mail.ru,Male,62524.770000,9378.7200
3,Lula,Sorrill,lsorrill3@hatena.ne.jp,Female,48000.770000,7200.1200
4,Blondell,Benard,bbenard4@admin.ch,Female,88638.490000,13295.7700
5,Jacquelyn,Fawdry,jfawdry5@addthis.com,Female,96620.000000,14493.0000
6,Kurt,Dugue,kdugue6@istockphoto.com,Male,105041.020000,15756.1500
7,Cordelie,Bloxsom,cbloxsom7@state.gov,Woman,236589.680000,35488.4500
8,Katinka,Renzo,krenzo8@thetimes.co.uk,Woman,194301.630000,29145.2400
9,Ketty,Pakeman,kpakeman9@chicagotribune.com,Female,239219.050000,35882.8575


### Step 3: Fix negative incomes

In [20]:
df["income"] = df["income"].abs()
df["tax_15"] = df["tax_15"].abs()

print("Negative incomes remaining:", (df["income"] < 0).sum())

Negative incomes remaining: 0


### Step 4: Standardize `gender` with Judgment

Category Man, men, Male, Women, Woman, Femal are self-evident. However, catogries Non-binary, Agender, Bigender, Genderfluid, Genderqueer, and Polygender. We should not flatten those into Male/Female, since it would misrepresent people and destroy real info.

In [21]:
gender_map = {
  "Man": "Male", "Men": "Male",
  "Women": "Female", "Woman": "Female",
}

df["gender"] = df["gender"].replace(gender_map)
print(df["gender"].value_counts())

gender
Female         474
Male           422
Non-binary      21
Genderfluid     19
Polygender      18
Genderqueer     17
Bigender        16
Agender         13
Name: count, dtype: int64


### Step 5: Normalize income (min-max)
rescaling coluns in to [0.1] using (value-min)/(max-min). This will make income comparable to other columns regardless of its large raw scale. So keep the original but add a normalized column.

In [ ]:
min = df["income"].min()
max = df["income"].max()
range = (max - min)
df["income_norm"] = (df["income"] - min) / (range)

In [42]:
print("Income range:", round(min, 2), "to", round(max, 2))

Income range: 15245.38 to 326986.67


In [43]:
print("Normalized range:", round(df["income_norm"].min(), 3), "to", round(df["income_norm"].max(), 3))
df[["income", "income_norm"]].head(20)

Normalized range: 0.0 to 1.0


,income,income_norm
0,123072.340000,0.346
1,160766.666667,0.467
2,62524.770000,0.152
3,48000.770000,0.105
4,88638.490000,0.235
5,96620.000000,0.261
6,105041.020000,0.288
7,236589.680000,0.710
8,194301.630000,0.574
9,239219.050000,0.718


### Step 6: Final check and save

In [45]:
print("Missing values remaining:", df.isna().sum().sum())
print("Negative incomes remaining:\n", (df["income"] < 0).sum())

df.head(20)
df.to_csv("./CleanedData.csv", index=False)
print("Saved CleanedData.csv")

Missing values remaining: 0
Negative incomes remaining:
 0
Saved CleanedData.csv


In [46]:
df["gender"].value_counts()

gender
Female         474
Male           422
Non-binary      21
Genderfluid     19
Polygender      18
Genderqueer     17
Bigender        16
Agender         13
Name: count, dtype: int64

## Your turn (in-class activity)

1. Print the **five-number summary** of the cleaned `income` column (min, Q1,
   median, Q3, max).
2. How many people are in each gender category *after* cleaning?
3. Save your notebook, then push it to your repo's `Code Examples` folder:

   ```
   git add "Code Examples/Preprocessing.ipynb"
   git commit -m "Add preprocessing lab"
   git push
   ```

In [8]:

import pandas as pd
import numpy as np

df = pd.read_csv("./CleanedData.csv")
df.head()

def func(column_name): 
    min = np.min(df[column_name])
    q1 = np.percentile(df[column_name], 25)
    med = np.median(df[column_name])
    q3 = np.percentile(df[column_name], 75)
    max = np.max(df[column_name])

    print(f'------------ Name:{column_name} -------------')
    print(f"Min: {min}")
    print(f"Q1: {q1}")
    print(f"Median: {med}")
    print(f"Q3: {q3}")
    print(f"Max: {max}")

func("income")

df["gender"].value_counts()


------------ Name:income -------------
Min: 15245.38
Q1: 80435.7175
Median: 139509.475
Q3: 201023.5875
Max: 326986.6666666667


gender
Female         474
Male           422
Non-binary      21
Genderfluid     19
Polygender      18
Genderqueer     17
Bigender        16
Agender         13
Name: count, dtype: int64